<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fengyee/TorchLeet/blob/main/v4/generative/fid/fid.ipynb)

Check your work: `pip install -e python/` once from the repo root (on Colab: `!pip install -q "git+https://github.com/Fengyee/TorchLeet#subdirectory=python"`), then `from torchleet import check; check("fid", ...)`


# Fréchet Inception Distance (FID) from Features

**Difficulty**: 🟢 Easy

---

### Problem Statement

**FID** (Heusel et al., 2017) is the default metric for image generators. Embed $N$ real and $N$ generated images with an Inception-v3 network (2048-d pool features), fit a Gaussian to each feature set, and report the **squared Fréchet (2-Wasserstein) distance between the two Gaussians**:

$$\mathrm{FID} = \lVert \mu_1 - \mu_2 \rVert_2^2 + \operatorname{Tr}\!\Big(\Sigma_1 + \Sigma_2 - 2\,\big(\Sigma_1 \Sigma_2\big)^{1/2}\Big)$$

Lower is better; 0 means the two Gaussians coincide. The network is the easy part — the subtle part is the matrix square root: $\Sigma_1\Sigma_2$ is **not symmetric**, the covariances are often singular ($N <$ feature dim in quick evaluations), and the answer is a small difference of large traces, so naive implementations return complex numbers, NaNs or garbage in float32. Here you skip the network and implement the statistics and the distance on given feature vectors.

---

### Requirements

1. **`feature_statistics(feats)`** — `feats`: `(N, D)` (rows are samples) → `(mu, sigma)` with `mu`: `(D,)` and `sigma`: `(D, D)` the **unbiased** covariance (divide by $N - 1$), i.e. the same as `torch.cov(feats.T)` / `numpy.cov(feats, rowvar=False)`. Must work when $N < D$.
2. **`frechet_distance(mu1, sigma1, mu2, sigma2)`** → a Python `float` (a 0-dim tensor is also accepted):
   - convert all inputs to **float64** first, whatever their dtype;
   - compute $\operatorname{Tr}\big((\Sigma_1\Sigma_2)^{1/2}\big) = \sum_i \sqrt{\lambda_i}$, where $\lambda_i$ are the eigenvalues of the symmetric PSD matrix $M = \Sigma_1^{1/2}\,\Sigma_2\,\Sigma_1^{1/2}$, with $\Sigma_1^{1/2} = V \operatorname{diag}(\sqrt{w})V^\top$ from `torch.linalg.eigh(sigma1)`;
   - clamp negative eigenvalues (round-off) to 0 before every square root;
   - return $\lVert\mu_1-\mu_2\rVert^2 + \operatorname{Tr}\Sigma_1 + \operatorname{Tr}\Sigma_2 - 2\sum_i\sqrt{\lambda_i}$.
   The result must be symmetric in its arguments, and finite and non-negative (up to ~1e-6 round-off) even for singular covariances.

---

### Constraints

- ✅ `torch.linalg.eigh` / `torch.linalg.eigvalsh` on symmetric matrices.
- ❌ No `scipy.linalg.sqrtm`, `torchmetrics`, `pytorch-fid` or `clean-fid`.
- ❌ Don't call `eigh` / `eigvalsh` on $\Sigma_1\Sigma_2$ itself — it is not symmetric and those routines read only one triangle.
- ❌ Don't take an elementwise `sqrt` of a matrix and call it a matrix square root.

---

<details>
  <summary>💡 Hint</summary>

  - Two matrices related by $B = P A P^{-1}$ have the same eigenvalues. Write $\Sigma_1\Sigma_2 = \Sigma_1^{1/2}\big(\Sigma_1^{1/2}\Sigma_2\Sigma_1^{1/2}\big)\Sigma_1^{-1/2}$.
  - The square root of a matrix with eigenvalues $\lambda_i$ has eigenvalues $\sqrt{\lambda_i}$, and the trace is the sum of eigenvalues — you never need the matrix square root itself, only its trace.
  - Symmetrize $M \leftarrow (M + M^\top)/2$ before `eigvalsh`: products of symmetric matrices are only symmetric up to round-off.

</details>

---

In [ ]:
import math

import torch

torch.manual_seed(0)

In [ ]:
# Synthetic "Inception features": correlated Gaussians in D = 64 (no network, no downloads)
D = 64
_A = torch.randn(D, D, generator=torch.Generator().manual_seed(0)) / math.sqrt(D)   # mixing matrix


def real_features(n, seed):
    g = torch.Generator().manual_seed(seed)
    return torch.randn(n, D, generator=g) @ _A.T + 1.0


def fake_features(n, seed, shift=0.0, scale=1.0):
    """A 'generator' whose features are shifted and/or over-dispersed versions of the real ones."""
    g = torch.Generator().manual_seed(seed)
    return scale * (torch.randn(n, D, generator=g) @ _A.T) + 1.0 + shift


real = real_features(5000, seed=1)
print(f"real features {tuple(real.shape)}")

In [ ]:
def feature_statistics(feats):
    """
    Args:
        feats: (N, D) feature vectors, one row per image.

    Returns:
        mu:    (D,) mean.
        sigma: (D, D) unbiased covariance (divide by N - 1), same as torch.cov(feats.T).
    """
    # TODO
    raise NotImplementedError


def frechet_distance(mu1, sigma1, mu2, sigma2):
    """
    Fréchet distance between N(mu1, sigma1) and N(mu2, sigma2), computed in float64.

    Args:
        mu1, mu2:       (D,) means.
        sigma1, sigma2: (D, D) symmetric PSD covariances (possibly singular).

    Returns:
        float: ||mu1 - mu2||^2 + Tr(sigma1) + Tr(sigma2) - 2 * Tr((sigma1 @ sigma2)^{1/2}).
    """
    # TODO
    raise NotImplementedError

In [ ]:
# Test 1: statistics match torch.cov (unbiased), also with fewer samples than dimensions
mu, sigma = feature_statistics(real)
assert mu.shape == (D,) and sigma.shape == (D, D)
assert torch.allclose(mu, real.mean(0), atol=1e-5)
assert torch.allclose(sigma, torch.cov(real.T), atol=1e-4), "use the unbiased (N - 1) covariance"
few = real[:10]
assert torch.allclose(feature_statistics(few)[1], torch.cov(few.T), atol=1e-5)
print("Test 1 passed: feature_statistics")

In [ ]:
# Test 2: closed forms
mu64, s64 = mu.double(), sigma.double()
assert abs(frechet_distance(mu64, s64, mu64, s64)) < 1e-6, "identical statistics -> 0"

delta = torch.linspace(-1, 1, D, dtype=torch.float64)
d = frechet_distance(mu64, s64, mu64 + delta, s64)
assert abs(d - (delta @ delta).item()) < 1e-6, "a pure translation costs ||delta||^2"

a = torch.rand(D, dtype=torch.float64) + 0.1
b = torch.rand(D, dtype=torch.float64) + 0.1
d = frechet_distance(torch.zeros(D), torch.diag(a), torch.zeros(D), torch.diag(b))
assert abs(d - ((a.sqrt() - b.sqrt()) ** 2).sum().item()) < 1e-8, "diagonal case: sum (sqrt(a) - sqrt(b))^2"

mu_f, s_f = feature_statistics(fake_features(5000, seed=2, shift=0.3, scale=1.2))
assert abs(frechet_distance(mu, sigma, mu_f, s_f) - frechet_distance(mu_f, s_f, mu, sigma)) < 1e-8, "symmetric"
print("Test 2 passed: zero, translation, diagonal closed form, symmetry")

In [ ]:
# Test 3: random SPD matrices vs an independent reference, and the rank-deficient case
def reference_fid(mu1, s1, mu2, s2):
    """Different route: complex eigenvalues of the non-symmetric product, principal sqrt, real part."""
    mu1, s1, mu2, s2 = (t.double() for t in (mu1, s1, mu2, s2))
    tr = torch.linalg.eigvals(s1 @ s2).sqrt().real.sum()
    return ((mu1 - mu2) @ (mu1 - mu2) + s1.trace() + s2.trace() - 2 * tr).item()


g = torch.Generator().manual_seed(3)
for dim in (8, 32):
    A1, A2 = torch.randn(dim, 2 * dim, generator=g), torch.randn(dim, 2 * dim, generator=g)
    s1, s2 = A1 @ A1.T / dim, 3 * A2 @ A2.T / dim
    m1, m2 = torch.randn(dim, generator=g), torch.randn(dim, generator=g)
    got, want = frechet_distance(m1, s1, m2, s2), reference_fid(m1, s1, m2, s2)
    assert abs(got - want) < 1e-6 * max(1.0, want), (dim, got, want)

f1, f2 = real_features(20, seed=4), fake_features(20, seed=5, shift=0.5)   # N = 20 < D = 64
d = frechet_distance(*feature_statistics(f1), *feature_statistics(f2))
assert math.isfinite(d) and d > 0, f"rank-deficient covariances must still give a finite FID, got {d}"
print("Test 3 passed: matches the eigvals reference; finite for singular covariances")

In [ ]:
# FID reacts to both mean shifts and covariance mismatch
print("shift  scale   FID")
for shift, scale in ((0.0, 1.0), (0.1, 1.0), (0.3, 1.0), (0.0, 1.2), (0.0, 0.8), (0.3, 1.2)):
    fake = fake_features(5000, seed=6, shift=shift, scale=scale)
    print(f"{shift:5.1f}  {scale:5.1f}  {frechet_distance(mu, sigma, *feature_statistics(fake)):7.3f}")

In [ ]:
# FID is biased upward for small N: two samples from the SAME distribution still score > 0,
# and the bias shrinks roughly like 1/N (here D = 64; Inception's D = 2048 makes it much worse).
print("    N   FID(real sample A, real sample B)")
for n in (100, 400, 1600, 6400):
    d = frechet_distance(*feature_statistics(real_features(n, seed=10)),
                         *feature_statistics(real_features(n, seed=11)))
    print(f"{n:5d}   {d:.4f}")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("fid", feature_statistics=feature_statistics, frechet_distance=frechet_distance)